CÉLULA 0: Extração de Features

In [143]:
import re
import traceback
from pathlib import Path
from tempfile import TemporaryDirectory
import warnings

import mne
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
mne.set_log_level("WARNING")

# ============================================================
# CONFIGURAÇÃO
# ============================================================
PASTA_AD = r"C:\Users\tiago\Downloads\Dataset_EEG_Alzheimer\dataset_eeg_alzheimer"
PASTA_HC = r"C:\Users\tiago\Downloads\Dataset_EEG_Alzheimer\dataset_eeg_hc"

OUT_CSV_FULL = "eeg_features_brainlat_FULL.csv"
OUT_FAILS = "eeg_features_brainlat_falhas.csv"

# Bandas usadas na extração espectral
BANDS = {
    "Delta": (1, 4),
    "Theta": (4, 8),
    "Alpha": (8, 13),
    "Beta":  (13, 30),
    "Gamma": (30, 45),
}

# Frequências para o cálculo espectral total
FMIN = 1
FMAX = 45


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================
def extrair_subject_id(arq_path: Path) -> str:
    """Extrai o identificador do sujeito do nome do arquivo."""
    m = re.search(r"(sub-\d+)", arq_path.stem, flags=re.IGNORECASE)
    if m:
        return m.group(1)

    m = re.search(r"sub[-_]?(\d+)", arq_path.stem, flags=re.IGNORECASE)
    if m:
        return f"sub-{m.group(1)}"

    return arq_path.stem


def inferir_pais(arq_path: Path) -> str:
    """Inferência simples do país a partir do caminho do arquivo."""
    parts = {p.upper() for p in arq_path.parts}
    if "AR" in parts:
        return "Argentina"
    if "CL" in parts:
        return "Chile"
    return "Outro"


def carregar_raw_seguro(arq_path: Path):
    """
    Carrega arquivo .set com fallback para o .fdt quando necessário.
    """
    try:
        return mne.io.read_raw_eeglab(str(arq_path), preload=True, verbose=False)
    except Exception as e:
        msg = str(e).lower()
        if ("fdt" in msg) or ("not found" in msg):
            fdt_exato = arq_path.with_suffix(".fdt")
            if fdt_exato.exists():
                with TemporaryDirectory() as tmp:
                    tmp_p = Path(tmp)
                    set_tmp = tmp_p / arq_path.name
                    fdt_tmp = tmp_p / fdt_exato.name

                    set_tmp.write_bytes(arq_path.read_bytes())
                    fdt_tmp.write_bytes(fdt_exato.read_bytes())

                    return mne.io.read_raw_eeglab(
                        str(set_tmp),
                        preload=True,
                        verbose=False
                    )
        raise e


def extrair_metricas_eeg(data, sfreq):
    """
    Extrai apenas métricas espectrais estáveis.

    Removido intencionalmente:
    - Mobility
    - Complexity
    - Soma_Check

    Motivo:
    essas métricas eram derivadas do sinal agregado por média temporal
    e podiam capturar artefatos de referência, não fisiologia.
    """
    from mne.time_frequency import psd_array_welch

    data = np.asarray(data, dtype=float)

    if data.ndim != 2:
        raise ValueError(f"Esperado array 2D (canais x amostras), recebido shape={data.shape}")

    n_fft = min(256, data.shape[-1])

    # PSD por canal
    psds, freqs = psd_array_welch(
        data,
        sfreq=sfreq,
        fmin=FMIN,
        fmax=FMAX,
        n_fft=n_fft,
        verbose=False
    )

    # Média entre canais
    psd_media = np.mean(psds, axis=0)
    metodo_area = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

    # Potência por banda
    pots = {}
    for banda, (fmin, fmax) in BANDS.items():
        idx = (freqs >= fmin) & (freqs < fmax)
        if np.sum(idx) >= 2:
            pots[banda] = float(metodo_area(psd_media[idx], freqs[idx]))
        else:
            pots[banda] = 0.0

    total = sum(pots.values()) + 1e-12

    # Features espectrais principais
    f = {
        "Rel_Delta_mean": pots["Delta"] / total,
        "Rel_Theta_mean": pots["Theta"] / total,
        "Rel_Alpha_mean": pots["Alpha"] / total,
        "Rel_Beta_mean":  pots["Beta"] / total,
        "Rel_Gamma_mean": pots["Gamma"] / total,
    }

    # Razões espectrais
    f["Razao_Theta_Alpha"] = f["Rel_Theta_mean"] / (f["Rel_Alpha_mean"] + 1e-12)
    f["Razao_Theta_Beta"] = f["Rel_Theta_mean"] / (f["Rel_Beta_mean"] + 1e-12)

    # Entropia espectral
    psd_norm = psd_media / (np.sum(psd_media) + 1e-12)
    f["Spectral_Entropy"] = float(-np.sum(psd_norm * np.log2(psd_norm + 1e-12)))

    return f


# ============================================================
# EXTRAÇÃO
# ============================================================
dados_finais = []
falhas = []

for grupo, pasta in [("AD", PASTA_AD), ("HC", PASTA_HC)]:
    arquivos = list(Path(pasta).rglob("*.set"))
    print(f"\n📂 {grupo}: {len(arquivos)} arquivos.")

    for arq in arquivos:
        try:
            sujeito_id = extrair_subject_id(arq)
            pais = inferir_pais(arq)

            raw = carregar_raw_seguro(arq)

            # Mantém apenas canais EEG
            raw.pick_types(eeg=True, eog=False, ecg=False, stim=False, emg=False, misc=False)

            # Epoching fixo
            epochs = mne.make_fixed_length_epochs(
                raw,
                duration=4.0,
                preload=True,
                verbose=False
            )

            if len(epochs) == 0:
                raise RuntimeError("Arquivo carregado, mas nenhuma época foi gerada.")

            data = epochs.get_data()
            sfreq = float(raw.info["sfreq"])

            for i, epoca in enumerate(data):
                feat = extrair_metricas_eeg(epoca, sfreq)

                feat.update({
                    "subject_id": sujeito_id,
                    "label": grupo,
                    "Pais": pais,
                    "epoch_id": i,
                    "source_file": arq.name,
                })

                dados_finais.append(feat)

            print(f"  ✅ {sujeito_id} ({len(data)} épocas)")

        except Exception as e:
            falhas.append({
                "grupo": grupo,
                "arquivo": arq.name,
                "caminho": str(arq),
                "erro": str(e),
                "traceback": traceback.format_exc(),
            })
            print(f"  ❌ {arq.name} → {e}")


# ============================================================
# DATAFRAMES E SALVAMENTO
# ============================================================
df_full = pd.DataFrame(dados_finais)
df_falhas = pd.DataFrame(falhas)

if df_full.empty:
    raise RuntimeError(
        "Nenhuma época foi extraída. "
        "Verifique os caminhos e a integridade dos arquivos .set."
    )

# Reordena colunas para facilitar inspeção
cols_meta = ["subject_id", "label", "Pais", "epoch_id", "source_file"]
cols_feat = [
    "Rel_Delta_mean",
    "Rel_Theta_mean",
    "Rel_Alpha_mean",
    "Rel_Beta_mean",
    "Rel_Gamma_mean",
    "Razao_Theta_Alpha",
    "Razao_Theta_Beta",
    "Spectral_Entropy",
]

cols_final = [c for c in cols_meta + cols_feat if c in df_full.columns]
df_full = df_full[cols_final]

df_full.to_csv(OUT_CSV_FULL, index=False, encoding="utf-8-sig")
df_falhas.to_csv(OUT_FAILS, index=False, encoding="utf-8-sig")

print(f"\n✅ Salvo: {OUT_CSV_FULL}")
print(f"✅ Salvo: {OUT_FAILS}")


# ============================================================
# RESUMO FINAL
# ============================================================
print("\n" + "=" * 50)
print("  RESUMO DA EXTRAÇÃO")
print("=" * 50)
print(f"  Sujeitos totais : {df_full['subject_id'].nunique()}")
print(f"  Sujeitos AD     : {df_full[df_full['label'] == 'AD']['subject_id'].nunique()}")
print(f"  Sujeitos HC     : {df_full[df_full['label'] == 'HC']['subject_id'].nunique()}")
print(f"  Épocas totais   : {len(df_full):,}")
print(f"  Falhas          : {len(df_falhas)}")
print()
print("  Distribuição País × Classe:")
print(
    df_full.drop_duplicates("subject_id")
           .groupby(["Pais", "label"])
           .size()
           .unstack(fill_value=0)
           .to_string()
)
print("=" * 50)


📂 AD: 35 arquivos.
  ✅ sub-30001 (78 épocas)
  ✅ sub-30002 (80 épocas)
  ✅ sub-30004 (71 épocas)
  ✅ sub-30008 (78 épocas)
  ✅ sub-30009 (101 épocas)
  ✅ sub-30011 (35 épocas)
  ✅ sub-30012 (56 épocas)
  ✅ sub-30013 (92 épocas)
  ✅ sub-30015 (40 épocas)
  ✅ sub-30017 (62 épocas)
  ✅ sub-30018 (105 épocas)
  ✅ sub-30020 (86 épocas)
  ✅ sub-30022 (104 épocas)
  ✅ sub-30026 (99 épocas)
  ✅ sub-30029 (113 épocas)
  ✅ sub-30031 (103 épocas)
  ✅ sub-30003 (109 épocas)
  ✅ sub-30005 (145 épocas)
  ✅ sub-30006 (155 épocas)
  ✅ sub-30007 (126 épocas)
  ✅ sub-30010 (118 épocas)
  ✅ sub-30014 (136 épocas)
  ✅ sub-30016 (120 épocas)
  ✅ sub-30019 (131 épocas)
  ✅ sub-30021 (90 épocas)
  ✅ sub-30023 (41 épocas)
  ✅ sub-30024 (142 épocas)
  ✅ sub-30025 (151 épocas)
  ✅ sub-30027 (169 épocas)
  ✅ sub-30028 (144 épocas)
  ✅ sub-30030 (124 épocas)
  ✅ sub-30032 (140 épocas)
  ✅ sub-30033 (124 épocas)
  ✅ sub-30034 (145 épocas)
  ✅ sub-30035 (103 épocas)

📂 HC: 46 arquivos.
  ✅ sub-100012 (86 épocas)
 

CÉLULA 1 — Importações e Configurações

In [144]:
# ============================================================
# CÉLULA 1 — INSTALAÇÕES E IMPORTAÇÕES
# ============================================================
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import roc_auc_score, recall_score, f1_score
from xgboost import XGBClassifier
from sklearn.utils import shuffle

try:
    from neuroCombat import neuroCombat
    COMBAT_AVAILABLE = True
except ImportError:
    COMBAT_AVAILABLE = False

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
print("✅ Ambiente configurado metodologicamente.")

✅ Ambiente configurado metodologicamente.


CÉLULA 2 — Funções de Pré-Processamento Seguras (Isolamento)

In [145]:
# ============================================================
# CÉLULA 2 — ISOLAMENTO DE TRANSFORMAÇÕES
# Justificativa: Para aplicar ComBat ou Scaler de maneira correta,
# eles devem ser ajustados SOMENTE no Train e aplicados ao Test.
# ============================================================

def aplicar_params_combat(X_te, combat_out):
    """
    Função mock para aplicar parâmetros do ComBat ao Teste.
    NOTA: Como o pacote padrão 'neuroCombat' em Python não possui
    um método .transform(), em produção recomenda-se usar pacotes como 
    'neuroCombat-sklearn' ou re-implementar a normalização por batch.
    Para este pipeline metodológico, garantimos que a transformação não olhe o teste.
    """
    return X_te.copy()

def aplicar_standard_scaler(X_tr, X_te, cols):
    """
    Aplica Z-score isolado: Estatísticas do treino ajustam o teste.
    Evita a dupla normalização e garante alinhamento geométrico para o SVM.
    """
    scaler = StandardScaler()
    X_tr_scaled = X_tr.copy()
    X_te_scaled = X_te.copy()
    
    X_tr_scaled[cols] = scaler.fit_transform(X_tr[cols])
    X_te_scaled[cols] = scaler.transform(X_te[cols])
    
    return X_tr_scaled, X_te_scaled

CÉLULA 3 — O Motor Central: LOSO Correto e Blindado

In [146]:
# ============================================================
# CÉLULA 7 — LOSO CORRETO (sem leakage de pré-processamento)
#
# FEATURE_COLS: Mobility e Complexity REMOVIDAS.
# Diagnóstico: ambas derivadas de np.mean(data, axis=0) no
# script de extração. Arquivos HC pré-processados com referência
# média → mean(canais) ≈ 0 por construção algébrica → Mobility
# colapsa para ~2.7e-7 e Complexity para ~1.732 em TODOS os
# sujeitos HC (std=0.013 em 32 sujeitos — impossível como sinal
# biológico). 6 sujeitos AD com referência diferente têm
# Complexity < 1.0 e Mobility > 0.003, criando separação
# perfeita por artefato de referência, não por Alzheimer.
# Features espectrais (Rel_*, Razao_*, Spectral_Entropy)
# são invariantes à referência → biologicamente defensáveis.
# ============================================================

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, recall_score, f1_score
import numpy as np
import pandas as pd

# ── Features biologicamente válidas ─────────────────────────
# Removidas:
#   Rel_Delta_mean → std = 0.0 (coluna morta no CSV)
#   Mobility       → artefato de referência de eletrodo
#   Complexity     → derivada de Mobility, mesmo artefato
#   Soma_Check     → variável de diagnóstico, não feature EEG
FEATURE_COLS = [
    'Rel_Theta_mean',
    'Rel_Alpha_mean',
    'Rel_Beta_mean',
    'Rel_Gamma_mean',
    'Razao_Theta_Alpha',
    'Razao_Theta_Beta',
    'Spectral_Entropy',
]


def aplicar_standard_scaler(X_tr, X_te, cols):
    """Fit APENAS no treino, transform em ambos. Sem dupla normalização."""
    scaler   = StandardScaler()
    X_tr_out = X_tr.copy()
    X_te_out = X_te.copy()
    X_tr_out[cols] = scaler.fit_transform(X_tr[cols])
    X_te_out[cols] = scaler.transform(X_te[cols])
    return X_tr_out, X_te_out


def loso_correto(X_df, y_arr, groups_arr, modelo_base,
                 feature_cols=FEATURE_COLS, verbose=True):
    """
    Leave-One-Subject-Out sem vazamento de pré-processamento.

    Parâmetros
    ----------
    X_df         : pd.DataFrame — épocas × features
    y_arr        : array-like   — rótulo binário por época (0=HC, 1=AD)
    groups_arr   : array-like   — subject_id por época
    modelo_base  : estimador sklearn — clonado a cada fold
    feature_cols : lista de features usadas no treino/predição
    verbose      : bool — imprime progresso por fold

    Retorna
    -------
    pd.DataFrame com colunas [subject_id, real, proba, n_epocas]
    ou None se nenhum fold for válido.
    """

    # ── Validações iniciais ──────────────────────────────────
    X_df       = X_df.reset_index(drop=True)
    y_arr      = np.asarray(y_arr).ravel()
    groups_arr = np.asarray(groups_arr).ravel()

    if not (len(X_df) == len(y_arr) == len(groups_arr)):
        raise ValueError(
            f"Tamanhos incompatíveis: X={len(X_df)}, "
            f"y={len(y_arr)}, groups={len(groups_arr)}"
        )

    cols_ausentes = [c for c in feature_cols if c not in X_df.columns]
    if cols_ausentes:
        raise ValueError(f"Features ausentes no DataFrame: {cols_ausentes}")

    # ── Auditoria de variância antes de treinar ───────────────
    std_zero = [c for c in feature_cols if X_df[c].std() < 1e-8]
    if std_zero:
        raise ValueError(
            f"Features com variância zero detectadas (artefato): {std_zero}\n"
            f"Remova-as de FEATURE_COLS antes de continuar."
        )

    logo       = LeaveOneGroupOut()
    preds      = []
    skipped    = []
    n_subjects = len(np.unique(groups_arr))

    if verbose:
        print(f"   Iniciando {n_subjects} folds LOSO...\n")

    # ── Loop principal ───────────────────────────────────────
    for fold_i, (train_idx, test_idx) in enumerate(
            logo.split(X_df, y_arr, groups=groups_arr)):

        subj_id = groups_arr[test_idx][0]

        X_tr_raw = X_df.iloc[train_idx]
        X_te_raw = X_df.iloc[test_idx]
        y_tr     = y_arr[train_idx]
        y_te     = y_arr[test_idx]

        # Fold inválido: treino sem as duas classes
        if len(np.unique(y_tr)) < 2:
            skipped.append(subj_id)
            if verbose:
                print(f"  ⚠  Fold {fold_i+1:02d} [{subj_id}] "
                      f"ignorado — treino com classe única")
            continue

        # [1] Normalização: fit APENAS no treino
        X_tr_sc, X_te_sc = aplicar_standard_scaler(
            X_tr_raw, X_te_raw, feature_cols
        )

        # [2] Modelo clonado (estado virgem a cada fold)
        modelo = clone(modelo_base)
        modelo.fit(X_tr_sc[feature_cols], y_tr)

        # [3] Predição por época
        proba_epocas = modelo.predict_proba(
            X_te_sc[feature_cols]
        )[:, 1]

        # [4] Agregação por sujeito (média das épocas)
        proba_sujeito = float(np.mean(proba_epocas))
        label_sujeito = int(y_te[0])

        preds.append({
            "subject_id" : subj_id,
            "real"       : label_sujeito,
            "proba"      : proba_sujeito,
            "n_epocas"   : len(proba_epocas),
        })

        if verbose:
            acerto = "✓" if int(proba_sujeito >= 0.5) == label_sujeito else "✗"
            print(f"  {acerto}  [{fold_i+1:02d}/{n_subjects}] "
                  f"{subj_id:<18} | "
                  f"real={'AD' if label_sujeito else 'HC'} | "
                  f"proba={proba_sujeito:.3f} | "
                  f"n_épocas={len(proba_epocas)}")

    # ── Resultado final ──────────────────────────────────────
    if len(preds) == 0:
        print("  ⚠  Nenhuma predição gerada — todos os folds foram ignorados.")
        return None

    if skipped and verbose:
        print(f"\n  ⚠  {len(skipped)} fold(s) ignorado(s): {skipped}")

    df_preds = pd.DataFrame(preds)

    pred_bin = (df_preds["proba"] >= 0.5).astype(int)
    df_preds.attrs["auc"]  = roc_auc_score(df_preds["real"], df_preds["proba"])
    df_preds.attrs["sens"] = recall_score(df_preds["real"], pred_bin,
                                          pos_label=1, zero_division=0)
    df_preds.attrs["spec"] = recall_score(df_preds["real"], pred_bin,
                                          pos_label=0, zero_division=0)
    df_preds.attrs["f1"]   = f1_score(df_preds["real"], pred_bin,
                                      zero_division=0)

    return df_preds  # ← FORA do loop


print("✅ loso_correto() e aplicar_standard_scaler() definidas.")
print(f"   FEATURE_COLS ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print()
print("⚠️  Mobility e Complexity excluídas — artefato de referência de eletrodo.")
print("   Hjorth correto requer cálculo POR CANAL com referência homogênea.")
print("   Ver: Célula 1 → extrair_metricas_eeg() para correção futura.")

✅ loso_correto() e aplicar_standard_scaler() definidas.
   FEATURE_COLS (7): ['Rel_Theta_mean', 'Rel_Alpha_mean', 'Rel_Beta_mean', 'Rel_Gamma_mean', 'Razao_Theta_Alpha', 'Razao_Theta_Beta', 'Spectral_Entropy']

⚠️  Mobility e Complexity excluídas — artefato de referência de eletrodo.
   Hjorth correto requer cálculo POR CANAL com referência homogênea.
   Ver: Célula 1 → extrair_metricas_eeg() para correção futura.


CÉLULA 4 — Definição dos Modelos (Saneados)

In [147]:
# ============================================================
# CÉLULA 4 — MODELOS SANEADOS
# Justificativa: O RobustScaler extra foi removido, pois o Z-score
# já é feito no passo 2 do LOSO correto.
# O SVM_Linear agora é calibrado com CV explícito ou usamos SGD (Huber loss).
# ============================================================

modelos_corrigidos = {
    'RandomForest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=SEED, n_jobs=-1),
    
    # Calibração explícita e CV seguro
    'SVM_Calibrado': CalibratedClassifierCV(
        estimator=SVC(kernel='linear', C=1.0, class_weight='balanced', random_state=SEED),
        cv=5
    ),
    
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, 
                             eval_metric='logloss', random_state=SEED, n_jobs=-1)
}

In [148]:
# Célula 10 — carregamento do CSV
import pandas as pd

df_audit = pd.read_csv("eeg_features_brainlat_FULL.csv")

# NÃO redefine FEATURE_COLS aqui — já definido na Célula 7
# Rel_Delta_mean tem std=0.0 no dataset inteiro e foi excluída

required_cols = ["subject_id", "label"] + FEATURE_COLS
missing_cols  = [c for c in required_cols if c not in df_audit.columns]
if missing_cols:
    raise ValueError(f"Colunas ausentes no CSV: {missing_cols}")

print(f"✅ Dados carregados: {df_audit.shape}")
print(f"✅ Sujeitos únicos : {df_audit['subject_id'].nunique()}")
print(f"✅ Classes         : {df_audit['label'].value_counts().to_dict()}")
display(df_audit[required_cols].head())
display(df_audit[required_cols].head())

✅ Dados carregados: (7425, 13)
✅ Sujeitos únicos : 67
✅ Classes         : {'AD': 3716, 'HC': 3709}


,subject_id,label,Rel_Theta_mean,Rel_Alpha_mean,Rel_Beta_mean,Rel_Gamma_mean,Razao_Theta_Alpha,Razao_Theta_Beta,Spectral_Entropy
0,sub-30001,AD,0.108649,0.442601,0.228321,0.196991,0.245479,0.475862,3.714057
1,sub-30001,AD,0.092019,0.339313,0.304871,0.235093,0.271191,0.301827,3.989446
2,sub-30001,AD,0.143054,0.442571,0.216033,0.171372,0.323235,0.662188,3.679847
3,sub-30001,AD,0.157965,0.449118,0.251806,0.119586,0.351722,0.627328,3.513216
4,sub-30001,AD,0.196091,0.416084,0.213736,0.150764,0.471276,0.917444,3.558616


,subject_id,label,Rel_Theta_mean,Rel_Alpha_mean,Rel_Beta_mean,Rel_Gamma_mean,Razao_Theta_Alpha,Razao_Theta_Beta,Spectral_Entropy
0,sub-30001,AD,0.108649,0.442601,0.228321,0.196991,0.245479,0.475862,3.714057
1,sub-30001,AD,0.092019,0.339313,0.304871,0.235093,0.271191,0.301827,3.989446
2,sub-30001,AD,0.143054,0.442571,0.216033,0.171372,0.323235,0.662188,3.679847
3,sub-30001,AD,0.157965,0.449118,0.251806,0.119586,0.351722,0.627328,3.513216
4,sub-30001,AD,0.196091,0.416084,0.213736,0.150764,0.471276,0.917444,3.558616


CÉLULA 5 — Execução do Benchmarking Correcto

In [149]:
# ============================================================
# CÉLULA 5 — EXECUÇÃO BENCHMARKING (LOSO BLINDADO)
#
# Correção aplicada: batch_arr removido da chamada.
# loso_correto() não recebe batch — a assinatura é:
#   loso_correto(X_df, y_arr, groups_arr, modelo_base,
#                feature_cols, verbose)
# ComBat está ausente por design (ver análise de leakage).
# ============================================================

from sklearn.metrics import roc_auc_score, recall_score

# ── Construção das variáveis de entrada ──────────────────────
X_full_df = df_audit[FEATURE_COLS].copy().reset_index(drop=True)
y_full    = (df_audit['label'] == 'AD').astype(int).values
groups    = df_audit['subject_id'].values

# ── Validação rápida antes de entrar no loop ─────────────────
assert len(X_full_df) == len(y_full) == len(groups), \
    "X, y e groups com tamanhos diferentes — checar df_audit"

assert all(c in X_full_df.columns for c in FEATURE_COLS), \
    "Uma ou mais FEATURE_COLS ausentes em df_audit"

print("=" * 60)
print(f"  Dataset: {len(groups):,} épocas | "
      f"{len(set(groups))} sujeitos | "
      f"{int(y_full.sum())} AD-épocas | "
      f"{int((1-y_full).sum())} HC-épocas")
print(f"  Features: {len(FEATURE_COLS)} → {FEATURE_COLS}")
print("=" * 60)
print()

# ── Benchmarking ─────────────────────────────────────────────
resultados_seguros = {}

for nome, clf in modelos_corrigidos.items():

    print(f"▶ Modelo: {nome}")

    df_res = loso_correto(
        X_df        = X_full_df,
        y_arr       = y_full,
        groups_arr  = groups,
        modelo_base = clf,
        feature_cols = FEATURE_COLS,
        verbose     = True        # muda para False para saída compacta
    )

    # ── Checagem de segurança antes de usar df_res ───────────
    if df_res is None or df_res.empty:
        print(f"  ❌ {nome}: loso_correto() retornou vazio ou None — "
              f"checar indentação da função e tamanho de preds\n")
        continue

    # ── Métricas (nível de sujeito) ──────────────────────────
    df_res['pred_bin'] = (df_res['proba'] >= 0.5).astype(int)

    auc  = roc_auc_score(df_res['real'], df_res['proba'])
    sens = recall_score(df_res['real'], df_res['pred_bin'],
                        pos_label=1, zero_division=0)
    spec = recall_score(df_res['real'], df_res['pred_bin'],
                        pos_label=0, zero_division=0)

    resultados_seguros[nome] = {
        'auc'     : round(auc,  4),
        'sens'    : round(sens, 4),
        'spec'    : round(spec, 4),
        'df_preds': df_res,
    }

    print(f"\n  ✅ {nome:>20}: "
          f"AUC={auc:.4f} | Sens={sens:.4f} | Spec={spec:.4f}")
    print("-" * 60 + "\n")

# ── Tabela resumo ─────────────────────────────────────────────
if resultados_seguros:
    print("=" * 60)
    print("  RESULTADO FINAL — LOSO POR SUJEITO")
    print("=" * 60)
    print(f"  {'Modelo':<22} {'AUC':>7} {'Sens':>7} {'Spec':>7}")
    print(f"  {'-'*22} {'-'*7} {'-'*7} {'-'*7}")
    for nome, r in resultados_seguros.items():
        print(f"  {nome:<22} {r['auc']:>7.4f} "
              f"{r['sens']:>7.4f} {r['spec']:>7.4f}")
    print("=" * 60)
else:
    print("⚠️  Nenhum modelo produziu resultado válido.")

  Dataset: 7,425 épocas | 67 sujeitos | 3716 AD-épocas | 3709 HC-épocas
  Features: 7 → ['Rel_Theta_mean', 'Rel_Alpha_mean', 'Rel_Beta_mean', 'Rel_Gamma_mean', 'Razao_Theta_Alpha', 'Razao_Theta_Beta', 'Spectral_Entropy']

▶ Modelo: RandomForest
   Iniciando 67 folds LOSO...

  ✓  [01/67] sub-10001          | real=HC | proba=0.001 | n_épocas=141
  ✓  [02/67] sub-100010         | real=HC | proba=0.001 | n_épocas=150
  ✓  [03/67] sub-100011         | real=HC | proba=0.000 | n_épocas=151
  ✓  [04/67] sub-100012         | real=HC | proba=0.125 | n_épocas=86
  ✓  [05/67] sub-100014         | real=HC | proba=0.000 | n_épocas=131
  ✓  [06/67] sub-100015         | real=HC | proba=0.008 | n_épocas=102
  ✓  [07/67] sub-100016         | real=HC | proba=0.000 | n_épocas=138
  ✓  [08/67] sub-100017         | real=HC | proba=0.453 | n_épocas=147
  ✓  [09/67] sub-100018         | real=HC | proba=0.028 | n_épocas=90
  ✓  [10/67] sub-10002          | real=HC | proba=0.001 | n_épocas=91
  ✓  [11/67] sub-

CÉLULA 6 — Permutation Test Agrupado (O Fim do P=1.0)

In [150]:
# ============================================================
# CÉLULA 6 — PERMUTATION TEST CORRETO POR SUJEITO
# Justificativa: Permutar épocas individuais cria dados irreais.
# Devemos permutar no nível do SUJEITO e aplicar o LOSO Blindado.
# ============================================================

def permute_by_group(y_arr, groups_arr):
    """Embaralha os rótulos de forma consistente para todas as épocas do mesmo sujeito."""
    sujeitos_unicos = np.unique(groups_arr)
    y_sujeitos = [y_arr[groups_arr == s][0] for s in sujeitos_unicos]
    y_sujeitos_shuffled = shuffle(y_sujeitos, random_state=None)
    
    dict_map = dict(zip(sujeitos_unicos, y_sujeitos_shuffled))
    return np.array([dict_map[g] for g in groups_arr])

n_perm = 100
aucs_nulos = []
modelo_teste = modelos_corrigidos['RandomForest']

print("\n🕵️ Iniciando Y-Randomization validado metodologicamente...")

# AUC real computado anteriormente
auc_real = resultados_seguros['RandomForest']['auc']

for i in range(n_perm):
    y_perm = permute_by_group(y_full, groups)
    df_res_perm = loso_correto(X_full_df, y_perm, groups, modelo_teste, df_audit, FEATURE_COLS)
    auc_nulo = roc_auc_score(df_res_perm['real'], df_res_perm['proba'])
    aucs_nulos.append(auc_nulo)

p_valor_correto = (np.sum(np.array(aucs_nulos) >= auc_real) + 1) / (n_perm + 1)

print(f"AUC Real       : {auc_real:.4f}")
print(f"AUC Médio Nulo : {np.mean(aucs_nulos):.4f}")
print(f"P-Valor        : {p_valor_correto:.4f}")

if p_valor_correto < 0.05:
    print("✅ Padrão Biológico Comprovado! (AUC reflete sinal real)")
else:
    print("🚨 Modelo não atingiu significância. Possível viés restante.")


🕵️ Iniciando Y-Randomization validado metodologicamente...


ValueError: Features ausentes no DataFrame: ['subject_id', 'label', 'Pais', 'epoch_id', 'source_file', 'Rel_Delta_mean']